# Smart City AI Project's EDA

## Objective:

The goal is to analyze, clean, prepare, and feature engineering the dataset

# Setep

Import the necessary libraries for data processing of the dataset

In [32]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn - preprocessing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# Sklearn - models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Sklearn - evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

Libraries imported successfully!


# Load Processed Data

In [33]:
city_df = pd.read_csv("../data/raw/smart_city_csvs/city_traffic_accidents.csv")


# City DF Data Overview

Analyze the structure, data types, and completeness of the dataset

In [34]:
# Display the rows of the dataset
city_df.head()

# Show dataset structure with the columns names and data types
city_df.info()

# Summary of the statistics for numerical columns
city_df.describe()

# Check for missing values in each column (Important)
city_df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  str    
 1   Source                 500000 non-null  str    
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  str    
 4   End_Time               500000 non-null  str    
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                280022 non-null  float64
 8   End_Lng                280022 non-null  float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            500000 non-null  str    
 11  Street                 499346 non-null  str    
 12  City                   499972 non-null  str    
 13  County                 500000 non-null  str    
 14  State                  500000 non-null  str    

ID                            0
Source                        0
Severity                      0
Start_Time                    0
End_Time                      0
Start_Lat                     0
Start_Lng                     0
End_Lat                  219978
End_Lng                  219978
Distance(mi)                  0
Description                   0
Street                      654
City                         28
County                        0
State                         0
Zipcode                     118
Country                       0
Timezone                    476
Airport_Code               1514
Weather_Timestamp          7842
Temperature(F)            10700
Wind_Chill(F)            129188
Humidity(%)               11371
Pressure(in)               9170
Visibility(mi)            11482
Wind_Direction            11382
Wind_Speed(mph)           36902
Precipitation(in)        142366
Weather_Condition         11300
Amenity                       0
Bump                          0
Crossing

## Data Quality Results

The dataset contains 500,000 records and 46 features. The target variable (Severity) has no missing values, but there are several features contain missing data, especially the End_lat & End_lng that has around 44% missing.


In [35]:
city_df.describe()

,Severity,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in)
count,500000.000000,500000.000000,500000.000000,280022.000000,280022.000000,500000.000000,489300.000000,370812.000000,488629.000000,490830.000000,488518.000000,463098.000000,357634.000000
mean,2.212386,36.205450,-94.706816,36.269349,-95.712652,0.558359,61.657263,58.223946,64.802873,29.536751,9.090389,7.689107,0.008308
std,0.487531,5.076753,17.383787,5.269770,18.096908,1.719821,19.013075,22.385876,22.857251,1.007469,2.703371,5.283195,0.104710
min,1.000000,24.555269,-124.497585,24.571240,-124.497421,0.000000,-89.000000,-89.000000,1.000000,0.120000,0.000000,0.000000,0.000000
25%,2.000000,33.400517,-117.217072,33.461970,-117.747945,0.000000,49.000000,43.000000,48.000000,29.370000,10.000000,4.600000,0.000000
50%,2.000000,35.831195,-87.792388,36.191048,-88.022662,0.029000,64.000000,62.000000,67.000000,29.860000,10.000000,7.000000,0.000000
75%,2.000000,40.081910,-80.351962,40.176147,-80.243654,0.462000,76.000000,75.000000,84.000000,30.030000,10.000000,10.400000,0.000000
max,4.000000,48.993996,-67.113167,48.995141,-67.109242,138.910004,136.400000,136.000000,100.000000,58.630000,100.000000,131.000000,10.020000


## Datetime Conversion

Looking at hints, this is to help with datetime columns contained inconsistent formats, which caused parsing errors (Used Google too)

In [36]:

city_df['Start_Time'] = pd.to_datetime(city_df['Start_Time'], errors='coerce')
city_df['End_Time'] = pd.to_datetime(city_df['End_Time'], errors='coerce')

In [37]:
city_df[['Start_Time', 'End_Time']].isnull().sum()

Start_Time    47737
End_Time      47737
dtype: int64

# Temporal Feature Engineering (Used both Hint #2 & Google)

Needed to extract features such as hour of day, day of week, and seasonal patterns to capture temporal trends. Added indicators for rush hour periods and accident duration

Doing the following temporal features like rush hour, weekday/weekend, and duration since time patterns strongly impact traffic severity.

In [38]:
#### Temoral Feature Engineering ###

# Extract hour of day
city_df['hour'] = city_df['Start_Time'].dt.hour

# Extract day of week (0 = Monday, 6 = Sunday)
city_df['day_of_week'] = city_df['Start_Time'].dt.dayofweek

# Extract month (seasonality)
city_df['month'] = city_df['Start_Time'].dt.month

# Identify weekends
city_df['is_weekend'] = (city_df['day_of_week'] >= 5).astype(int)


### Rush Hour Features ###

# Morning rush (7–9 AM)
city_df['is_morning_rush'] = city_df['hour'].between(7, 9).astype(int)

# Evening rush (4–7 PM)
city_df['is_evening_rush'] = city_df['hour'].between(16, 19).astype(int)

# Combined rush hour indicator
city_df['is_rush_hour'] = (
    city_df['is_morning_rush'] | city_df['is_evening_rush']
).astype(int)


### Duration Feature ###

city_df['duration_min'] = (
    (city_df['End_Time'] - city_df['Start_Time'])
    .dt.total_seconds() / 60
)

# Cap extreme values (max 24 hours)
city_df['duration_min'] = city_df['duration_min'].clip(0, 1440)

# Fill missing time-based features with -1 (unknown)
city_df['hour'].fillna(-1, inplace=True)
city_df['day_of_week'].fillna(-1, inplace=True)
city_df['month'].fillna(-1, inplace=True)

0         10.0
1         10.0
2          8.0
3          6.0
4          3.0
          ... 
499995     4.0
499996     5.0
499997     3.0
499998     5.0
499999    11.0
Name: month, Length: 500000, dtype: float64

In [ ]:
# Handle missing and invalid temporal values doing the following code / Used Google for help because I kept getting errors after the above codes
# - Replace infinite values if there was any per google
# - Fill missing values with -1 to indicate unknown time per google
# - Convert to nullable integer type for consistency and robustness

In [40]:
# Check what is still causing issues
print(city_df['month'].isnull().sum())
print(np.isinf(city_df['month']).sum())

# Replace any infinite values just in case
city_df['month'] = city_df['month'].replace([np.inf, -np.inf], np.nan)

# Fill missing values
city_df['month'] = city_df['month'].fillna(-1)

# Convert safely
city_df['month'] = city_df['month'].astype('Int64')

47737
0


In [41]:
city_df['month'].dtype
city_df['month'].head()

0    10
1    10
2     8
3     6
4     3
Name: month, dtype: Int64

In [42]:
city_df['hour'] = city_df['hour'].replace([np.inf, -np.inf], np.nan).fillna(-1).astype('Int64')
city_df['day_of_week'] = city_df['day_of_week'].replace([np.inf, -np.inf], np.nan).fillna(-1).astype('Int64')

# Weather Feature Engineering / Hint #3

The weather conditions play a signifcant role in accident severity. Missing weather data is not random and may indicate unavailable sensor data or API issues.

Will create features to capture weather availability, hazardous conditions, and simplified weather categories.

In [43]:
# Define weather columns
weather_cols = [
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)'
]

# Weather Availability Flag

# 1 = all weather data present, 0 = some missing
city_df['weather_data_available'] = (
    city_df[weather_cols].notna().all(axis=1)
).astype(int)


# Hazard Features

# Freezing temperatures (<= 32°F)
city_df['is_freezing'] = (city_df['Temperature(F)'] <= 32).astype(int)

# Low visibility (< 2 miles)
city_df['low_visibility'] = (city_df['Visibility(mi)'] < 2).astype(int)


# Weather Condition Categorization

def categorize_weather(condition):
    if pd.isna(condition):
        return 'unknown'
    
    condition = str(condition).lower()
    
    if any(w in condition for w in ['clear', 'fair']):
        return 'clear'
    elif any(w in condition for w in ['cloud', 'overcast']):
        return 'cloudy'
    elif any(w in condition for w in ['rain', 'drizzle', 'shower']):
        return 'rain'
    elif any(w in condition for w in ['snow', 'sleet', 'ice', 'wintry']):
        return 'snow_ice'
    elif any(w in condition for w in ['fog', 'mist', 'haze', 'smoke']):
        return 'low_visibility'
    elif any(w in condition for w in ['thunder', 'storm']):
        return 'storm'
    else:
        return 'other'


# Apply grouping
city_df['weather_group'] = city_df['Weather_Condition'].apply(categorize_weather)

In [44]:
# Per Google, I need to do the following after fill numerica weather columns after creating the features

# Instead of just filling weather data, the below creates availiability flags and engineered hazard based faetures like freezing and low visiliby (Per Google)

for col in weather_cols:
    city_df[col] = city_df[col].fillna(city_df[col].median())